# 📊 Aşama 1: Veri Keşfi ve Ön İşleme

**Türkçe E-Ticaret Yorumları — Duygu Analizi & Metin Özetleme**

Bu notebook'ta:
1. Kaggle'dan Hepsiburada veri seti indirilecek
2. Veri keşfi (EDA) yapılacak — sınıf dağılımı, kelime istatistikleri, grafikler
3. Türkçe NLP ön işleme uygulanacak
4. Temizlenmiş veri `data/hepsiburada_clean.csv` olarak kaydedilecek

> ⚠️ **Google Colab kullanıyorsanız**, önce GitHub deposunu klonlayın (aşağıdaki ilk hücre).

## 🔧 0) Colab Kurulumu (sadece Colab'da çalıştırın)

In [ ]:
# Colab'da çalışıyorsanız bu hücreyi çalıştırın:
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    # GitHub'dan kodu klonla
    !git clone https://github.com/erncanyildirim/metin-ozetleme-ve-duygu-analizi.git
    %cd metin-ozetleme-ve-duygu-analizi
    # Bağımlılıkları kur
    !pip install -q -r requirements.txt
    print('✅ Colab kurulumu tamamlandı!')
else:
    print('Lokalde çalışıyorsunuz — kurulum hatırlatması: pip install -r requirements.txt')

## 🔑 1) Kaggle API Kurulumu

Veri setini indirebilmek için Kaggle API anahtarınız gerekli:

1. https://www.kaggle.com adresine gidip hesap açın (ücretsiz)
2. Sağ üstte profil → **Account** → **API** bölümüne gidin
3. **Create New API Token** butonuna basın → `kaggle.json` indirilecek
4. Aşağıdaki hücrede dosya yükleme butonu çıkacak, `kaggle.json`'ı yükleyin

In [ ]:
import os

if IS_COLAB:
    from google.colab import files
    print('📤 kaggle.json dosyasını yükleyin:')
    uploaded = files.upload()
    
    # ~/.kaggle/kaggle.json'a taşı
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print('✅ Kaggle API hazır!')
else:
    # Lokalde: ~/.kaggle/kaggle.json'ın mevcut olduğunu varsay
    kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_path):
        print(f'✅ Kaggle yapılandırması bulundu: {kaggle_path}')
    else:
        print(f'⚠️  {kaggle_path} bulunamadı. Lütfen kaggle.json dosyasını oraya kopyalayın.')

## 📥 2) Veri Setini İndirme

In [ ]:
# Proje modüllerini import et
from src.data_loader import download_dataset, load_data, clean_data, add_sentiment_labels

# 1) İndir
csv_path = download_dataset()

In [ ]:
# 2) Pandas ile oku
df = load_data(csv_path)
df.head()

## 🔍 3) Keşifsel Veri Analizi (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Temel bilgiler
print(f'📊 Boyut: {df.shape}')
print(f'📊 Sütunlar: {df.columns.tolist()}')
print(f'\n📊 Veri tipleri:')
print(df.dtypes)
print(f'\n📊 Eksik değerler:')
print(df.isnull().sum())

In [ ]:
# Veriyi temizle
df = clean_data(df)
df = add_sentiment_labels(df)
df.head()

In [ ]:
# Sınıf dağılımı - Pasta grafiği (rapor için Şekil 4.1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = df['sentiment'].value_counts()
colors = {'pozitif': '#28a745', 'nötr': '#ffc107', 'negatif': '#dc3545'}
color_list = [colors.get(s, '#888') for s in sentiment_counts.index]

# Pasta
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index, 
            colors=color_list, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Duygu Sınıfı Dağılımı', fontsize=14, fontweight='bold')

# Bar
axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=color_list)
axes[1].set_title('Yorum Sayıları', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Yorum sayısı')
for i, v in enumerate(sentiment_counts.values):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('rapor/sekil_4_1_sinif_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Yorum uzunluk dağılımı (Şekil 4.2)
text_col = 'review' if 'review' in df.columns else df.select_dtypes('object').columns[0]
df['word_count'] = df[text_col].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['word_count'], bins=50, color='#4a90e2', edgecolor='black')
axes[0].set_title('Yorum Uzunluğu Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Kelime sayısı')
axes[0].set_ylabel('Yorum sayısı')
axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Ortalama: {df['word_count'].mean():.1f}")
axes[0].legend()

# Sınıf bazlı boxplot
df_plot = df[df['word_count'] < df['word_count'].quantile(0.95)]
sns.boxplot(data=df_plot, x='sentiment', y='word_count', 
            palette=colors, ax=axes[1])
axes[1].set_title('Sınıf Başına Uzunluk', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Kelime sayısı')

plt.tight_layout()
plt.savefig('rapor/sekil_4_2_uzunluk_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Uzunluk istatistikleri:')
print(df['word_count'].describe())

## 🧹 4) Ön İşleme Uygulama

In [ ]:
from src.preprocessing import preprocess_dataframe, get_word_freq, get_text_stats

# Ön işleme uygula
df = preprocess_dataframe(df)
df[['review', 'clean_text', 'sentiment']].head(10) if 'review' in df.columns else df[['clean_text', 'sentiment']].head(10)

In [ ]:
# En sık kelimeleri görselleştir (Şekil 4.3)
freq = get_word_freq(df, column='clean_text', top_n=20)

plt.figure(figsize=(10, 6))
freq.plot(kind='barh', color='#667eea')
plt.gca().invert_yaxis()
plt.title('En Sık 20 Kelime (Ön İşleme Sonrası)', fontsize=14, fontweight='bold')
plt.xlabel('Frekans')
plt.tight_layout()
plt.savefig('rapor/sekil_4_3_en_sik_kelimeler.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sınıf bazlı en sık kelimeler
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, sentiment in enumerate(['pozitif', 'nötr', 'negatif']):
    subset = df[df['sentiment'] == sentiment]['clean_text']
    words = ' '.join(subset.astype(str)).split()
    common = Counter(words).most_common(15)
    w, c = zip(*common)
    
    axes[i].barh(w, c, color=colors.get(sentiment, '#888'))
    axes[i].invert_yaxis()
    axes[i].set_title(f'{sentiment.upper()} sınıfında en sık kelimeler', fontsize=12, fontweight='bold')
    
plt.tight_layout()
plt.savefig('rapor/sekil_4_3b_sinif_bazli_kelimeler.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 5) Temizlenmiş Veriyi Kaydet

In [ ]:
# Boş clean_text'leri çıkar
df = df[df['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)

output_path = 'data/hepsiburada_clean.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'✅ Temizlenmiş veri kaydedildi: {output_path}')
print(f'📊 Toplam {len(df):,} yorum')

# Colab'daysak Google Drive'a da yedek alalım (isteğe bağlı)
if IS_COLAB:
    print('\n💡 Drive\'a kaydetmek isterseniz:')
    print('   from google.colab import drive')
    print('   drive.mount("/content/drive")')
    print('   df.to_csv("/content/drive/MyDrive/hepsiburada_clean.csv", index=False)')

## 📋 6) Özet İstatistikleri (Rapor için)

Aşağıdaki sayıları **Bölüm 4.3 (Veri Seti İstatistikleri)** kısmına yazacaksın.

In [ ]:
print('=' * 60)
print('  📊 RAPOR İÇİN ÖZET İSTATİSTİKLER (Bölüm 4.3)')
print('=' * 60)
print(f'  Toplam yorum sayısı       : {len(df):,}')
print(f'  Ortalama yorum uzunluğu   : {df["word_count"].mean():.1f} kelime')
print(f'  Medyan yorum uzunluğu     : {df["word_count"].median():.0f} kelime')
print(f'  Maksimum yorum uzunluğu   : {df["word_count"].max()} kelime')
print()
print('  Sınıf Dağılımı:')
for sent, count in df['sentiment'].value_counts().items():
    pct = count / len(df) * 100
    print(f'    {sent:10s}: {count:>7,} ({pct:5.1f}%)')

print('\n✅ Notebook 1 tamamlandı! Sonraki adım: 02_baseline_model.ipynb')